# Last.fm batch recommender

Run `recommend_album()` over a subset of `albums.csv` and write results to
`datasets/lastfm_recommendations_<subset>_<strategy>.csv`.

Each recommendation is one row with a `rank` column, plus Last.fm listener counts and
top tags for the seed and recommended albums (`seed_tags`, `rec_tags`).

**Resume:** re-run the batch cell — albums that already have a `status` are skipped.
Older CSVs without tag columns still load; missing fields are filled with `NA`.

Set `LASTFM_API_KEY` in `lastfm-recommender/.env` (see `.env.example`).


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from lastfm_albums import (
    format_album_tags,
    get_album_listeners_and_tags,
    recommend_album,
    set_request_delay,
)


In [ ]:
DATA_DIR = Path("../datasets")
SOURCE_PATH = DATA_DIR / "albums.csv"

TRACK_SELECTION = "top_listener"  # top_listener | random | top_n_tracks
TOP_N = 3            # seed tracks for top_n_tracks
N_RECS = 5           # recommendations per album
FETCH_FLOOR = 20     # similar tracks fetched per seed (more API calls when higher)
REQUEST_DELAY_SEC = 0.7
RANDOM_SEED = 45     # base seed for random track selection
SAVE_EVERY = 10      # checkpoint CSV every N albums


## Subset & output path

Pick a subset key and optional `MAX_ALBUMS` cap. The output filename follows
`lastfm_recommendations_{subset}_{strategy}.csv`.


In [ ]:
SUBSETS = {
    "all": lambda df: df,
    "3plus": lambda df: df.loc[df["review_count"] > 2],
    "test": lambda df: df.sample(250, random_state=RANDOM_SEED),
    "2k": lambda df: df.sample(2000, random_state=RANDOM_SEED),
}

SUBSET = "all"
MAX_ALBUMS = None  # e.g. 10 for a dry run

In [ ]:
albums = SUBSETS[SUBSET](pd.read_csv(SOURCE_PATH))
if MAX_ALBUMS is not None:
    albums = albums.head(MAX_ALBUMS)

strategy_key = (
    f"top_n_tracks_{TOP_N}" if TRACK_SELECTION == "top_n_tracks" else TRACK_SELECTION
)
OUTPUT_PATH = DATA_DIR / f"lastfm_recommendations_{SUBSET}_{strategy_key}.csv"
print(f"Subset: {SUBSET} ({len(albums):,} albums)")
print(f"Strategy: {TRACK_SELECTION}")
print(f"Output: {OUTPUT_PATH}")


## Resume from checkpoint

Load existing output (if any), normalize dtypes, and compute the pending queue.


In [6]:
OUTPUT_COLS = [
    "album_id", "artist", "album", "review_count",
    "strategy", "status", "error", "seed_track", "seed_listeners", "seed_tags",
    "rec_artist", "rec_album", "score", "rank", "rec_listeners", "rec_tags",
]

results = pd.read_csv(OUTPUT_PATH) if OUTPUT_PATH.exists() else pd.DataFrame(columns=OUTPUT_COLS)

In [ ]:
for col in OUTPUT_COLS:
    if col not in results.columns:
        results[col] = pd.NA

for col in ("strategy", "status", "error", "seed_track", "seed_tags", "rec_artist", "rec_album", "rec_tags"):
    results[col] = results[col].astype("string")
results["score"] = pd.to_numeric(results["score"], errors="coerce")
for col in ("rank", "seed_listeners", "rec_listeners"):
    results[col] = pd.to_numeric(results[col], errors="coerce").astype("Int64")

processed_ids = set(
    results.loc[
        results["status"].notna() & (results["status"].astype(str).str.strip() != ""),
        "album_id",
    ]
)
pending = albums[~albums["album_id"].isin(processed_ids)]

print(f"Pending: {len(pending):,} / {len(albums):,}")
results.head(2)


## Run batch

Interrupt safely anytime — progress is checkpointed every `SAVE_EVERY` albums.


In [ ]:
set_request_delay(REQUEST_DELAY_SEC)

new_rows: list[dict] = []
since_save = 0

for _, row in tqdm(pending.iterrows(), total=len(pending), desc="Last.fm batch"):
    album_seed = RANDOM_SEED + int(row["album_id"])
    base = {
        "album_id": row["album_id"],
        "artist": row["artist"],
        "album": row["album"],
        "review_count": row["review_count"],
        "strategy": TRACK_SELECTION,
    }

    try:
        seed, seed_track, recs, _ = recommend_album(
            row["album"],
            artist=row["artist"],
            n_recs=N_RECS,
            fetch_floor=FETCH_FLOOR,
            track_selection=TRACK_SELECTION,
            top_n=TOP_N,
            random_seed=album_seed,
            clear_cache=False,
        )
        seed_tags = format_album_tags(seed.get("tags") or [])
        if recs.empty:
            new_rows.append({
                **base,
                "status": "no_results",
                "error": pd.NA,
                "seed_track": pd.NA,
                "seed_listeners": seed.get("listeners", pd.NA),
                "seed_tags": seed_tags,
                "rec_artist": pd.NA,
                "rec_album": pd.NA,
                "score": pd.NA,
                "rank": pd.NA,
                "rec_listeners": pd.NA,
                "rec_tags": pd.NA,
            })
        else:
            for rank, (_, rec) in enumerate(recs.iterrows(), start=1):
                if rank > N_RECS:
                    break
                try:
                    rec_listeners, rec_tag_list = get_album_listeners_and_tags(
                        rec["artist"], rec["album"]
                    )
                    rec_tags = format_album_tags(rec_tag_list)
                except Exception:
                    rec_listeners = pd.NA
                    rec_tags = pd.NA
                new_rows.append({
                    **base,
                    "status": "ok",
                    "error": pd.NA,
                    "seed_track": seed_track["name"] if seed_track else pd.NA,
                    "seed_listeners": seed.get("listeners", pd.NA),
                    "seed_tags": seed_tags,
                    "rec_artist": rec["artist"],
                    "rec_album": rec["album"],
                    "score": rec["similarity_score"],
                    "rank": rank,
                    "rec_listeners": rec_listeners,
                    "rec_tags": rec_tags,
                })
    except Exception as exc:
        new_rows.append({
            **base,
            "status": "error",
            "error": str(exc)[:500],
            "seed_track": pd.NA,
            "seed_listeners": pd.NA,
            "seed_tags": pd.NA,
            "rec_artist": pd.NA,
            "rec_album": pd.NA,
            "score": pd.NA,
            "rank": pd.NA,
            "rec_listeners": pd.NA,
            "rec_tags": pd.NA,
        })

    since_save += 1
    if since_save >= SAVE_EVERY:
        results = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
        new_rows = []
        results.to_csv(OUTPUT_PATH, index=False)
        since_save = 0

if new_rows:
    results = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
results.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

## Quick look


In [ ]:
print(results["status"].value_counts(dropna=False))

has_tags = (results["status"] == "ok") & results["seed_tags"].notna()
cols = ["artist", "album", "seed_track", "rank", "rec_album", "rec_artist", "score", "seed_tags", "rec_tags"]
results.loc[has_tags, cols].head(3)


## Retry failed albums (optional)

Uncomment and run to drop `error` / `no_results` rows and rebuild `pending`,
then re-run the batch cell.


In [ ]:
# retry_ids = results.loc[results["status"].isin(["error", "no_results"]), "album_id"].unique()
# results = results[~results["album_id"].isin(retry_ids)]
# pending = albums[~albums["album_id"].isin(
#     results.loc[results["status"].notna(), "album_id"].unique()
# )]
# print(f"Reset {len(retry_ids):,} albums; pending: {len(pending):,}")
